# 02. Análisis Exploratorio de Datos (EDA)

Este notebook realiza un análisis exploratorio completo del dataset Boston Housing para entender patrones, relaciones y características importantes.

## Objetivos
- Analizar distribuciones de variables
- Identificar relaciones entre variables
- Detectar valores atípicos
- Visualizar correlaciones
- Generar insights para modelado

In [ ]:
# Importación de librerías
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Configuración de visualización
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 8)
%matplotlib inline

In [ ]:
# Cargar datos limpios
df = pd.read_csv('../data/boston_housing_clean.csv')
print(f"Dataset cargado: {df.shape}")
print(f"Columnas: {list(df.columns)}")

## 1. Análisis de Distribuciones

In [ ]:
# Histogramas de todas las variables numéricas
fig, axes = plt.subplots(4, 4, figsize=(20, 16))
axes = axes.ravel()

for i, col in enumerate(df.columns):
    axes[i].hist(df[col], bins=30, alpha=0.7, edgecolor='black')
    axes[i].set_title(f'Distribución de {col}')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Frecuencia')

plt.tight_layout()
plt.show()

In [ ]:
# Boxplots para detectar outliers
fig, axes = plt.subplots(4, 4, figsize=(20, 16))
axes = axes.ravel()

for i, col in enumerate(df.columns):
    axes[i].boxplot(df[col])
    axes[i].set_title(f'Boxplot de {col}')
    axes[i].set_ylabel(col)

plt.tight_layout()
plt.show()

## 2. Análisis de la Variable Objetivo (MEDV)

In [ ]:
# Análisis detallado de MEDV (precio de viviendas)
plt.figure(figsize=(15, 5))

# Histograma
plt.subplot(1, 3, 1)
plt.hist(df['medv'], bins=30, alpha=0.7, edgecolor='black')
plt.title('Distribución de Precios de Viviendas (MEDV)')
plt.xlabel('Precio ($1000s)')
plt.ylabel('Frecuencia')

# Boxplot
plt.subplot(1, 3, 2)
plt.boxplot(df['medv'])
plt.title('Boxplot de MEDV')
plt.ylabel('Precio ($1000s)')

# Q-Q plot para normalidad
plt.subplot(1, 3, 3)
stats.probplot(df['medv'], dist="norm", plot=plt)
plt.title('Q-Q Plot de MEDV')

plt.tight_layout()
plt.show()

# Estadísticas descriptivas
print("=== ESTADÍSTICAS DE MEDV ===")
print(df['medv'].describe())

In [ ]:
# Test de normalidad
from scipy.stats import shapiro, jarque_bera

# Shapiro-Wilk test
shapiro_stat, shapiro_p = shapiro(df['medv'])
print(f"Shapiro-Wilk Test: estadístico={shapiro_stat:.4f}, p-valor={shapiro_p:.4f}")

# Jarque-Bera test
jb_stat, jb_p = jarque_bera(df['medv'])
print(f"Jarque-Bera Test: estadístico={jb_stat:.4f}, p-valor={jb_p:.4f}")

if shapiro_p > 0.05:
    print("✅ La distribución de MEDV parece normal (p > 0.05)")
else:
    print("❌ La distribución de MEDV no es normal (p ≤ 0.05)")

## 3. Matriz de Correlaciones

In [ ]:
# Calcular matriz de correlación
correlation_matrix = df.corr()

# Crear heatmap de correlaciones
plt.figure(figsize=(14, 12))
sns.heatmap(correlation_matrix, 
            annot=True, 
            cmap='coolwarm', 
            center=0,
            square=True,
            fmt='.2f',
            cbar_kws={'shrink': 0.8})
plt.title('Matriz de Correlación - Boston Housing Dataset')
plt.tight_layout()
plt.show()

In [ ]:
# Identificar correlaciones más fuertes con MEDV
medv_correlations = correlation_matrix['medv'].sort_values(key=abs, ascending=False)
print("=== CORRELACIONES CON MEDV (PRECIO) ===")
for var, corr in medv_correlations.items():
    if var != 'medv':
        print(f"{var}: {corr:.3f}")
        
# Correlaciones más fuertes en general
print("\n=== CORRELACIONES MÁS FUERTES EN EL DATASET ===")
# Obtener correlaciones fuera de la diagonal
corr_pairs = []
for i in range(len(correlation_matrix.columns)):
    for j in range(i+1, len(correlation_matrix.columns)):
        corr_val = correlation_matrix.iloc[i, j]
        corr_pairs.append((correlation_matrix.columns[i], 
                          correlation_matrix.columns[j], 
                          abs(corr_val), 
                          corr_val))

# Ordenar por valor absoluto de correlación
corr_pairs.sort(key=lambda x: x[2], reverse=True)

print("Top 10 correlaciones más fuertes:")
for i, (var1, var2, abs_corr, corr) in enumerate(corr_pairs[:10]):
    print(f"{i+1}. {var1} - {var2}: {corr:.3f}")

## 4. Análisis de Relaciones Clave

In [ ]:
# Variables más correlacionadas con MEDV
top_features = ['rm', 'lstat', 'ptratio', 'indus', 'tax', 'nox', 'crim']

# Scatter plots de variables más importantes
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.ravel()

for i, feature in enumerate(top_features):
    axes[i].scatter(df[feature], df['medv'], alpha=0.6, s=30)
    axes[i].set_xlabel(feature)
    axes[i].set_ylabel('MEDV (Precio)')
    axes[i].set_title(f'{feature} vs MEDV\nCorrelación: {correlation_matrix.loc[feature, "medv"]:.3f}')
    
    # Agregar línea de tendencia
    z = np.polyfit(df[feature], df['medv'], 1)
    p = np.poly1d(z)
    axes[i].plot(df[feature], p(df[feature]), "r--", alpha=0.8)

# Scatter plot adicional para boston-housing-specific
axes[7].scatter(df['age'], df['medv'], alpha=0.6, s=30)
axes[7].set_xlabel('AGE')
axes[7].set_ylabel('MEDV (Precio)')
axes[7].set_title(f'AGE vs MEDV\nCorrelación: {correlation_matrix.loc["age", "medv"]:.3f}')

plt.tight_layout()
plt.show()

## 5. Análisis de Variables Categóricas

In [ ]:
# Análisis de CHAS (variable binaria)
plt.figure(figsize=(15, 5))

# Distribución de CHAS
plt.subplot(1, 3, 1)
chas_counts = df['chas'].value_counts()
plt.pie(chas_counts.values, labels=['No limita con río', 'Limita con río'], autopct='%1.1f%%')
plt.title('Distribución de CHAS')

# Precio promedio por CHAS
plt.subplot(1, 3, 2)
chas_price = df.groupby('chas')['medv'].mean()
plt.bar(['No limita', 'Limita'], chas_price.values)
plt.title('Precio promedio por CHAS')
plt.ylabel('Precio medio ($1000s)')

# Boxplot de precios por CHAS
plt.subplot(1, 3, 3)
df.boxplot(column='medv', by='chas', ax=plt.gca())
plt.title('Distribución de precios por CHAS')
plt.suptitle('')
plt.xlabel('CHAS')
plt.ylabel('MEDV')

plt.tight_layout()
plt.show()

# Estadísticas por CHAS
print("=== ESTADÍSTICAS POR CHAS ===")
print(df.groupby('chas')['medv'].describe())

## 6. Identificación de Patrones Geográficos

In [ ]:
# Análisis de patrones espaciales (usando variables proxy)
plt.figure(figsize=(15, 10))

# Crear bins para DIS (distancia al empleo)
df['dis_category'] = pd.cut(df['dis'], bins=3, labels=['Cerca', 'Media', 'Lejos'])

# Precio vs distancia al empleo
plt.subplot(2, 2, 1)
dis_price = df.groupby('dis_category')['medv'].mean()
plt.bar(dis_price.index, dis_price.values)
plt.title('Precio promedio por distancia al empleo')
plt.ylabel('Precio medio ($1000s)')

# Tasa de criminalidad vs precio
plt.subplot(2, 2, 2)
plt.scatter(df['crim'], df['medv'], alpha=0.6)
plt.xlabel('Tasa de criminalidad')
plt.ylabel('Precio ($1000s)')
plt.title('Criminalidad vs Precio')

# Tamaño de vivienda vs precio
plt.subplot(2, 2, 3)
plt.scatter(df['rm'], df['medv'], alpha=0.6)
plt.xlabel('Número promedio de habitaciones')
plt.ylabel('Precio ($1000s)')
plt.title('Tamaño de vivienda vs Precio')

# Edad vs precio
plt.subplot(2, 2, 4)
plt.scatter(df['age'], df['medv'], alpha=0.6)
plt.xlabel('Proporción de viviendas antiguas')
plt.ylabel('Precio ($1000s)')
plt.title('Antigüedad vs Precio')

plt.tight_layout()
plt.show()

## 7. Resumen de Insights

In [ ]:
# Generar resumen de insights clave
insights = {
    'correlaciones_fuertes': [],
    'distribuciones_no_normales': [],
    'outliers_significativos': [],
    'patrones_interesantes': []
}

# Correlaciones más fuertes con MEDV
strong_corrs = medv_correlations[abs(medv_correlations) > 0.5]
insights['correlaciones_fuertes'] = list(strong_corrs.index[1:])  # Excluir MEDV consigo mismo

# Variables con distribución no normal
for col in df.columns:
    _, p_value = stats.jarque_bera(df[col])
    if p_value < 0.05:
        insights['distribuciones_no_normales'].append(col)

# Guardar insights
import json
with open('../reports/eda_insights.json', 'w') as f:
    json.dump(insights, f, indent=2)

print("=== INSIGHTS CLAVE DEL ANÁLISIS EXPLORATORIO ===")
print(f"\n1. Variables más correlacionadas con el precio:")
for var in insights['correlaciones_fuertes']:
    corr_val = correlation_matrix.loc[var, 'medv']
    print(f"   - {var}: {corr_val:.3f}")

print(f"\n2. Variables con distribución no normal: {len(insights['distribuciones_no_normales'])}")
print(f"3. Total de variables analizadas: {len(df.columns)}")
print(f"4. Tamaño del dataset: {df.shape[0]} registros")

print("\n✅ Análisis exploratorio completado")
print("📁 Insights guardados en: ../reports/eda_insights.json")

## Conclusiones del EDA

### Principales Hallazgos:

1. **Variables más importantes para predecir precio:**
   - LSTAT (% clase baja): correlación negativa fuerte (-0.74)
   - RM (habitaciones): correlación positiva fuerte (0.70)
   - PTRATIO (ratio alumno-maestro): correlación negativa (-0.51)

2. **Distribución de precios:**
   - No sigue distribución normal
   - Rango: $5,000 - $50,000 (en miles de 1970s)
   - Valores atípicos en el extremo superior

3. **Relaciones interesantes:**
   - Criminalidad vs Precio: relación negativa
   - Tamaño de vivienda vs Precio: relación positiva
   - Proximidad al río (CHAS): impacto limitado en precio

4. **Preparación para modelado:**
   - Considerar transformaciones para normalizar distribuciones
   - Manejo de outliers en variables como CRIM y ZN
   - Escalado de variables debido a diferentes rangos

Los datos están listos para la preparación y modelado en el siguiente notebook.